STEP 1 — Install Libraries

In [ ]:
!pip install transformers torch accelerate bitsandbytes sentencepiece gradio PyPDF2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.0 MB/s eta 0:00:00


STEP 2 — Import Libraries

In [ ]:
import io, torch
import PyPDF2
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import files

STEP 3 — Login to HuggingFace

In [ ]:
HF_TOKEN = "Hugging_Face_Token"
login(token=HF_TOKEN)

STEP 4 — Load the AI Model

In [ ]:
MODEL_ID  = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,token = HF_TOKEN,quantization_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_use_double_quant = True,
        bnb_4bit_quant_type       = "nf4",
        bnb_4bit_compute_dtype    = torch.float16),
    device_map = {"": 0})

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

STEP 5 — The Optimize Function

In [ ]:
def optimize_resume(resume_text):
  prompt = f"""You are a world-class resume writer and career coach with 15+ years of experience helping candidates get hired at top companies.
Your job is to fully optimize the resume below and make it outstanding.
FOLLOW ALL 15 RULES STRICTLY:
1.  Keep every section heading EXACTLY as in the original — do not rename, remove, or add any section.
2.  Keep all personal facts UNCHANGED — full name, email, phone, college name, CGPA, company names, project names, certification names.
3.  Rewrite OBJECTIVE completely — make it powerful, specific, and confident. Show who the candidate is, what they offer, and what they want.
4.  SKILLS — keep all original skills, group them cleanly, and add 3 to 5 relevant skills that naturally match the candidate's background.
5.  PROJECTS — keep every project title exactly as written. Rewrite each description using the format: Action → Task → Result.
6.  Every project bullet must start with a strong past-tense action verb like Engineered, Designed, Built, Developed, Implemented, Deployed.
7.  Add measurable outcomes to projects wherever possible — example: "improved accuracy by 30%" or "reduced load time by 40%".
8.  INTERNSHIP — keep company name exactly. Rewrite every duty as an achievement bullet showing what was done and what impact it had.
9.  ACHIEVEMENTS — make each point sharp, specific, and impactful using bullet points. Add context — what, where, and why it matters.
10. STRENGTHS — rewrite as professional competencies. Never use vague words. Example: instead of "Hard worker" write "Ability to deliver complex projects under tight deadlines".
11. Keep EDUCATION and HOBBIES sections WORD-FOR-WORD exactly as in the original. Do not change a single word.
12. Never use weak words like "helped", "assisted", "worked on", "did", "made". Replace with powerful verbs only.
13. Every bullet point must be one crisp, clear, impactful sentence. No long paragraphs.
14. Do NOT invent or add fake experience, fake skills, fake certifications, or fake companies.
15. Output ONLY the optimized resume. No introduction, no explanation, no comments, no notes. Start directly with the resume.
ORIGINAL RESUME:
{resume_text}
OPTIMIZED RESUME:"""
  inputs = tokenizer(
      f"[INST] {prompt.strip()} [/INST]", return_tensors = "pt", truncation = True, max_length = 3500).to("cuda")
  with torch.no_grad():
    output = model.generate(**inputs,max_new_tokens = 1500,do_sample = False,repetition_penalty = 1.15,pad_token_id = tokenizer.eos_token_id)
  result = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:],skip_special_tokens = True).strip()
  for tag in ["OPTIMIZED RESUME:", "REWRITTEN RESUME:", "[INST]","[/INST]", "Note:", "Here is", "Here's", "OUTPUT:"]:
    result = result.replace(tag, "").strip()
  return result

STEP 6 — Upload Resume & Get Output

In [ ]:
print("UPLOAD YOUR RESUME — PDF or TEXT FILE")
uploaded    = files.upload()
resume_text = ""
for fname, content in uploaded.items():
  if fname.lower().endswith(".pdf"):
    reader      = PyPDF2.PdfReader(io.BytesIO(content))
    resume_text = "\n".join(p.extract_text() or "" for p in reader.pages).strip()
    print(f"PDF loaded — {len(resume_text)} characters extracted")
  else:
    resume_text = content.decode("utf-8").strip()
    print(f"Text file loaded — {len(resume_text)} characters")
if resume_text:
  print("\n AI is optimizing your resume... please wait (1–2 min)\n")
  result = optimize_resume(resume_text)
  print(result)
else:
  print("File could not be read. Please try again with a PDF or .txt file.")

UPLOAD YOUR RESUME — PDF or TEXT FILE


Saving Resume.pdf to Resume (2).pdf
PDF loaded — 1723 characters extracted

 AI is optimizing your resume... please wait (1–2 min)

MANASA MS
+91 9865907741 | manasagowda1846@gmail.com

Professional Summary:
Confident Final Year B.Com Student with a hands-on internship experience in the automobile industry and a strong foundation in accounting and business operations. Dedicated to securing an entry-level position in accounting, finance, or sales to leverage analytical and communication skills.

Education:
Bachelor of Commerce, St. Joseph's First Grade College, Hassan
CGPA: 8.7/10, 2023-2026

Pre-University – Commerce, BGS PU College, Hassan
Score: 85.5%, 2022-2023

SSLT – Science Stream, Pradhana High School, Hassan
Score: 74.4%, 2020-2021

Internship Experience:
Intern, Hoysala Honda (Sri VLS Motors LLP)
January 2026 – February 2026

* Analyzed departmental data to enhance operational efficiency
* Understood real-time business processes and implemented reporting systems
* Coordinated 